In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import scienceplots
plt.style.use(['science', 'nature', 'grid'])

import numpy as np
from scipy.interpolate import interp1d
import pandas as pd

from LoadMoments import *
from LoadTSV import *

In [ ]:
name_cons = "../out/out_cons.tsv"
name_prim = "../out/out_prim.tsv"

In [ ]:
data_cons = read_solution_file(name_cons)
        
times = np.array([data_cons[ts]["t"] for ts in sorted(data_cons.keys())])
timesteps = np.array(sorted(data_cons.keys()))

time_query = len(data_cons)
sol = interpolate_solution(data_cons, time_query, times, timesteps)
x, U = sol["x"], sol["u"]

data_prim = read_solution_file(name_prim)
time_query = len(data_prim)
sol_prim = interpolate_solution(data_prim, time_query, times, timesteps)
P = sol_prim["u"]

In [ ]:
rho = U[:, 0]
v = U[:, 1] / rho
theta = 1 / (3 * rho) * (U[:, 4] + U[:, 7] + U[:, 9] - rho * v**2)
p = rho * theta

In [ ]:
fig, axs = plt.subplots()
axs.plot(x, rho, label=r"$\rho$")
twx = axs.twinx(); twx.plot(x, v, label=r"$v$")
axs.plot(x, p, label=r"$p$")
axs.grid()
fig.tight_layout(); fig.show()

# Compare to 1D-3D Slab-Geometry

In [ ]:
# Plotting
colors = ['blue', 'orange', 'green', 'red', 'purple', 'brown']
linestyles = ['-', '--', ':', '-.', (0, (3, 1, 1, 1)), (0, (5, 1))]

# Parameter
M = 4
closure = "ExtGram"
Kn = 1.0
source_string = "relaxation_source"
T_end = 0.3
base_tree_level = 10
polydeg = 1
rho_L = 7.0
v_L1 = 0.0
v_L2 = 0.0
v_L3 = 0.0
theta_L = 1.0
rho_R = 1.0
v_R1 = 0.0
v_R2 = 0.0
v_R3 = 0.0
theta_R = 1.0

# Solution files
n_angle_pairs = 4

marker = ['o', 's', '^', 'x', 'd']

# Kn and relaxation for solution files
Kn = 1.0
source = 'relaxation_source'
name = lambda anglepairnumber, Kn, source_string: f"../out/1D3D/1D3D_angles/gram_solution_M{M}_closure{closure}_Kn{Kn}_source{source_string}_T_end{T_end}_rho_L{rho_L}_rho_R{rho_R}_v_L1{v_L1}_v_L2{v_L2}_v_L3{v_L3}_v_R1{v_R1}_v_R2{v_R2}_v_R3{v_R3}_theta_L{theta_L}_theta_R{theta_R}_base_tree_level{base_tree_level}_polydeg{polydeg}_anglepairnumber{anglepairnumber}_cons.tsv"

In [ ]:
fig, axs = plt.subplots(figsize=(6.0 * 3/4, 2.5))
twx = axs.twinx()
twx.grid(False)

axs.set_xlim(-1, 1)
axs.set_xlabel('x')
axs.set_ylabel('$\\rho, p$')
twx.set_ylabel('$v$')

# Stable angles
for i in range(n_angle_pairs - 1):
    solution_file = name(i, Kn, source)
    data = read_solution_file(solution_file)
    
    times = np.array([data[ts]["t"] for ts in sorted(data.keys())])
    timesteps = np.array(sorted(data.keys()))
    
    time_query = 21 
    sol = interpolate_solution(data, time_query, times, timesteps)
    x, U = sol["x"], sol["u"]

    rho = U[:, 0]
    v = U[:, 1] / U[:, 0]
    theta = 1 / (3 * rho) * (U[:, 2] + 2 * U[:, 3] - rho * v**2) 
    p = rho * theta

    axs.plot(x, rho, color=colors[0], linestyle=linestyles[i])
    twx.plot(x, v, color=colors[1], linestyle=linestyles[i])
    axs.plot(x, p, color=colors[2], linestyle=linestyles[i])
# Legend and Layout
axs.plot([], [], color=colors[0], lw=2, label='$\\rho$')
axs.plot([], [], color=colors[1], lw=2, label='$v$')
axs.plot([], [], color=colors[2], lw=2, label='$p$')




# And the full 1D-3D solution
name_cons = "../out/out_cons.tsv"
name_prim = "../out/out_prim.tsv"

data_cons = read_solution_file(name_cons)
        
times = np.array([data_cons[ts]["t"] for ts in sorted(data_cons.keys())])
timesteps = np.array(sorted(data_cons.keys()))

time_query = len(data_cons)
sol = interpolate_solution(data_cons, time_query, times, timesteps)
x, U = sol["x"], sol["u"]

data_prim = read_solution_file(name_prim)
time_query = len(data_prim)
sol_prim = interpolate_solution(data_prim, time_query, times, timesteps)
P = sol_prim["u"]

rho = U[:, 0]
v = U[:, 1] / rho
theta = 1 / (3 * rho) * (U[:, 4] + U[:, 7] + U[:, 9] - rho * v**2)
p = rho * theta

axs.plot(x, rho, '--o', color=colors[0])
twx.plot(x, v, '--o', color=colors[1])
axs.plot(x, p, '--o', color=colors[2])

[axs.plot([], [], color='black', linestyle=linestyles[i], lw=2, label=f'1D-3D (Slab) with $\\zeta_{{{i}}}$') for i in range(n_angle_pairs - 1)]
axs.plot([], [], '--o', color='black', lw=2, label=f'1D-3D (Full, Fibonacci angles)')

axs.legend(bbox_to_anchor=(1.3, 1.0), loc='upper center')

# Note: tight_layout often crushes manual insets, so we use it carefully or skip it
# fig.tight_layout() 
fig.savefig(f'1D3DFullComparison.pdf', bbox_inches='tight')
plt.show()

In [ ]:
M_vec = [4, 5, 6, 7, 8]

fig, axs = plt.subplots(figsize=(6.0 * 3/4, 2.5))
twx = axs.twinx()
twx.grid(False)

axs.set_xlim(-1, 1)
axs.set_xlabel('x')
axs.set_ylabel('$\\rho, p$')
twx.set_ylabel('$v$')
for i, M in enumerate(M_vec):
    # And the full 1D-3D solution
    name_cons = f"../out/1D3V_full/M={M}_cons.tsv"
    name_prim = f"../out/1D3V_full/M={M}_prim.tsv"

    data_cons = read_solution_file(name_cons)
            
    times = np.array([data_cons[ts]["t"] for ts in sorted(data_cons.keys())])
    timesteps = np.array(sorted(data_cons.keys()))

    time_query = len(data_cons)
    sol = interpolate_solution(data_cons, time_query, times, timesteps)
    x, U = sol["x"], sol["u"]

    data_prim = read_solution_file(name_prim)
    time_query = len(data_prim)
    sol_prim = interpolate_solution(data_prim, time_query, times, timesteps)
    P = sol_prim["u"]

    rho = U[:, 0]
    v = U[:, 1] / rho
    theta = 1 / (3 * rho) * (U[:, 4] + U[:, 7] + U[:, 9] - rho * v**2)
    p = rho * theta

    axs.plot(x, rho, color=colors[0], linestyle=linestyles[i])
    twx.plot(x, v, color=colors[1], linestyle=linestyles[i])
    axs.plot(x, p, color=colors[2], linestyle=linestyles[i])

    axs.plot([], [], color='black', linestyle=linestyles[i], lw=1, label=f'M={M}')

axs.legend(bbox_to_anchor=(1.3, 1.0), loc='upper center')

# Note: tight_layout often crushes manual insets, so we use it carefully or skip it
# fig.tight_layout() 
fig.savefig(f'1D3DFull_Moments.pdf', bbox_inches='tight')
plt.show()